# 02 — Probability Surface Construction

Cleans raw bid/ask prices into normalized (de-vigged) implied probabilities,
then builds a "probability surface" across outcome bucket, event date, and
time-to-resolution — the prediction-market analogue of a vol surface.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from probability_cleaning import build_clean_probability_table
from feature_engineering import add_time_to_resolution
import visualization as viz

pd.set_option("display.max_columns", 20)
%matplotlib inline

In [ ]:
markets_df = pd.read_csv("../data/raw/markets.csv")
price_df = pd.read_csv("../data/raw/price_history.csv", parse_dates=["timestamp"])
macro_df = pd.read_csv("../data/raw/macro_features.csv", parse_dates=["date"])

price_df.head()

## A. Implied probability extraction

Mid-price → normalize across outcomes within each (market, timestamp) so
probabilities sum to 1 (removing the overround / vig).

In [ ]:
clean_df = build_clean_probability_table(price_df)
clean_df.head(10)

In [ ]:
print("Average overround (vig) by market, before normalization:")
clean_df.groupby("market_id")["overround"].mean().sort_values(ascending=False)

Example outcome table for the most recent snapshot of one market
(mirrors the "Fed cuts" example table in the project spec):

In [ ]:
example_market = clean_df["market_id"].unique()[0]
latest = (
    clean_df[clean_df["market_id"] == example_market]
    .sort_values("timestamp")
    .groupby("outcome_name")
    .tail(1)[["outcome_name", "mid", "clean_prob"]]
    .rename(columns={"mid": "market_price", "clean_prob": "clean_probability"})
)
latest["market_price"] = (latest["market_price"] * 100).round(1).astype(str) + "%"
latest["clean_probability"] = (latest["clean_probability"] * 100).round(1).astype(str) + "%"
latest

## B. Probability surface: path into resolution, per outcome

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
viz.plot_probability_paths(clean_df, example_market, ax=ax)
plt.show()

In [ ]:
clean_df = add_time_to_resolution(clean_df, markets_df)
clean_df[["market_id", "outcome_name", "timestamp", "days_to_resolution", "clean_prob"]].head()

3D surface — one outcome bucket ("1 cut") across all simulated FOMC
cycles, x = days to resolution, y = market (chronological), z = probability.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
viz.plot_probability_surface_3d(clean_df, "1 cut", ax=ax)
plt.show()

## Persist cleaned + surfaced data for the modeling notebook

Note: we persist the *pre-merge* `clean_prob` table (without
`expiration_date`/`days_to_resolution`) — notebook 03's feature pipeline
adds those itself via `build_feature_table`. Keeping the persisted schema
minimal avoids duplicate-column collisions on re-merge.

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

persist_cols = ["market_id", "outcome_name", "timestamp", "bid", "ask", "volume",
                 "mid", "spread", "overround", "clean_prob", "trust_weight"]
clean_df[persist_cols].to_csv("../data/processed/clean_probabilities.csv", index=False)
print(f"Saved {clean_df.shape[0]} rows to data/processed/clean_probabilities.csv")